# Blood Donation Prediction System — Complete Analysis Notebook

**Project:** Blood Donation Prediction System  
**Authors:** Mihret Alemayehu, Abebech Nega  
**Institution:** Debre Berhan University  
**Instructor:** Petros Abebe  

---

## Purpose of This Notebook

This notebook is the **presentation version** of the full pipeline. It runs every step
of the analysis — from loading raw data to evaluating the trained model — and displays
all graphs **inline** directly under each code cell.

> **How to use:**  
> Run cells one by one (Shift+Enter) to walk through the analysis step by step.  
> Every chart will appear below the cell that creates it.

---

## Notebook Structure

| Section | Description |
|---|---|
| 1 | Environment Setup |
| 2 | Load Raw Data |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Missing Value Analysis |
| 5 | Data Cleaning (KNN Imputation) |
| 6 | Feature Engineering |
| 7 | Model Training (Random Forest) |
| 8 | Model Evaluation |
| 9 | Summary of Results |

---

> **Note on dual output:**  
> - `run_pipeline.py` saves all graphs as `.png` files in `reports/figures/` using `plt.savefig()`  
> - This notebook displays all graphs **inline** using `%matplotlib inline` + `plt.show()`  
> Both outputs are identical in content — this notebook is intended for classroom presentation.


---
## Section 1 — Environment Setup

We begin by importing all required libraries and configuring the display settings.  
`%matplotlib inline` ensures every chart appears directly in the notebook output area.


In [ ]:
# ── Standard library imports ──────────────────────────────────────────────────
import os
import sys
import json
import pickle
import warnings
from datetime import datetime

# ── Data science libraries ────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_recall_curve,
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── IMPORTANT: %matplotlib inline makes all plots appear in the notebook ──────
# This is what allows plt.show() to display charts directly under code cells.
# In run_pipeline.py we use matplotlib.use('Agg') + plt.savefig() instead.
%matplotlib inline

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')

# Set a consistent visual style for all charts in this notebook
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# ── Add project root to Python path so we can import from src/ ────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Folder paths (used throughout the notebook) ───────────────────────────────
RAW_CSV    = os.path.join(PROJECT_ROOT, 'data', 'raw', 'blood_donation.csv')
PROC_DIR   = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIG_DIR    = os.path.join(PROJECT_ROOT, 'reports', 'figures')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

# Create output folders if they do not exist yet
for d in [PROC_DIR, FIG_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Confirm environment is ready ──────────────────────────────────────────────
print('Environment ready.')
print(f'Python  : {sys.version.split()[0]}')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'seaborn : {sns.__version__}')
print(f'Project root: {PROJECT_ROOT}')


---
## Section 2 — Load Raw Data

We use the `DataLoader` utility class from `src/data/loader.py` to read the CSV.  
The `quick_summary()` method prints shape, missing values, and a preview of the data.


In [ ]:
# ── Import our custom DataLoader class from src/data/loader.py ───────────────
# This class handles CSV/Excel/Parquet loading and basic data summaries.
from src.data.loader import DataLoader

# Initialize the loader pointing to our raw data folder
loader = DataLoader(data_dir=os.path.join(PROJECT_ROOT, 'data', 'raw'))

# Load the blood donation CSV file into a pandas DataFrame
df_raw = loader.load_csv('blood_donation.csv')

# Print a quick summary: shape, data types, missing values, first 5 rows
loader.quick_summary(df_raw)


In [ ]:
# ── Detailed overview of all columns ─────────────────────────────────────────
# df.info() shows column names, data types, and how many values are non-null.
# This helps us identify which columns have missing data and what type they are.
print('=== DataFrame Info ===')
df_raw.info()


In [ ]:
# ── Statistical summary of numeric columns ────────────────────────────────────
# describe() returns count, mean, std, min, max, and quartile statistics.
# This gives us a quick sense of the range and distribution of each feature.
print('=== Descriptive Statistics ===')
df_raw.describe(include='all').T.round(2)


In [ ]:
# ── Preview the first 10 rows ─────────────────────────────────────────────────
# Seeing actual data rows helps identify formatting issues, unusual values,
# or columns that may need special handling.
print('=== First 10 Rows ===')
df_raw.head(10)


---
## Section 3 — Exploratory Data Analysis (EDA)

EDA helps us understand the dataset before any modeling.  
We explore:
- Class balance (how many donors will vs. won't donate)
- Feature distributions (histograms)
- Feature-class relationships (box plots)
- Correlations between features (heatmap)

> Each chart is displayed inline using `plt.show()` and also saved to
> `reports/figures/` using `fig.savefig()`. Both happen in the same cell.


### 3.1 — Target Variable Distribution

The target column tells us whether a donor will donate again:
- `0` = Won't Donate
- `1` = Will Donate

We want to see if the dataset is **balanced or imbalanced** — a large imbalance
means our model needs special handling (we use `class_weight='balanced'` in the Random Forest).


In [ ]:
# ── Check target class distribution ──────────────────────────────────────────
# Count how many records belong to each class (0 and 1).
target_col = 'Target'
target_counts = df_raw[target_col].value_counts().sort_index()

print('=== Target Distribution ===')
for label, count in target_counts.items():
    pct = count / len(df_raw) * 100
    label_name = 'Will Donate' if label == 1 else "Won't Donate"
    print(f'  {label_name} ({label}): {count:,}  ({pct:.1f}%)')

# ── Draw the bar chart ────────────────────────────────────────────────────────
# Red = Won't Donate (0), Blue = Will Donate (1)
fig, ax = plt.subplots(figsize=(6, 5))

bars = ax.bar(
    ["Won't Donate (0)", 'Will Donate (1)'],
    target_counts.values,
    color=['#C0392B', '#2980B9'],
    edgecolor='white',
    width=0.5,
)

# Add count labels on top of each bar so exact numbers are visible
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{int(bar.get_height()):,}",
        ha='center', va='bottom', fontsize=11,
    )

ax.set_title('Target Variable Distribution', fontsize=13)
ax.set_ylabel('Number of Donors')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# plt.savefig() → saves PNG to disk
# plt.show()    → displays the chart inline in this notebook
fig.savefig(os.path.join(FIG_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/target_distribution.png')


### 3.2 — Feature Distributions (Histograms)

Histograms show us the shape of each numeric feature's distribution:
- Is it **skewed** (long tail on one side) or symmetric?
- Are there **outliers** (very long tails)?
- Is the distribution **bimodal** (two separate peaks)?

Understanding distribution shape helps us decide whether to apply transformations
and informs our imputation strategy.


In [ ]:
# ── Histogram grid for all numeric features ───────────────────────────────────
# We create one histogram per numeric column arranged in a 3-column grid.
# bins=30 gives good granularity without too much noise.

numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in numeric_cols if c != target_col]

print(f'Numeric features ({len(feat_cols)}): {feat_cols}')

n = len(feat_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols  # calculate number of rows needed

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = np.array(axes).ravel()  # flatten to 1D array for easy indexing

for i, col in enumerate(feat_cols):
    # bins=30: number of histogram bars | alpha=0.85: slight transparency
    axes[i].hist(df_raw[col].dropna(), bins=30, color='#2980B9',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Count')

# Hide any unused subplot panels
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
fig.tight_layout()

fig.savefig(os.path.join(FIG_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/feature_distributions.png')


### 3.3 — Box Plots by Target Class

Box plots compare the distribution of each feature across the two target classes.
- The **box** spans from the 25th to 75th percentile (interquartile range)
- The **line inside** the box is the median value
- **Dots** outside the whiskers are statistical outliers

If the boxes for class 0 and class 1 are clearly separated, that feature is
likely **useful for predicting** donation behavior.


In [ ]:
# ── Box plots: one per feature, split by target class ─────────────────────────
# Red (class 0) = Won't Donate | Blue (class 1) = Will Donate
# Large visual separation between the two boxes = strong predictive feature.

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = np.array(axes).ravel()

for i, col in enumerate(feat_cols):
    sns.boxplot(
        x=target_col, y=col, data=df_raw,
        palette={0: '#C0392B', 1: '#2980B9'},
        ax=axes[i],
    )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Target (0=Won't Donate, 1=Will Donate)")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions by Target Class', fontsize=14, y=1.01)
fig.tight_layout()

fig.savefig(os.path.join(FIG_DIR, 'boxplots_by_target.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/boxplots_by_target.png')


### 3.4 — Pearson Correlation Heatmap

The correlation heatmap shows how strongly each pair of features is linearly related:
- Values close to **+1.0** = strong positive correlation (both increase together)
- Values close to **-1.0** = strong negative correlation (one increases as the other decreases)
- Values close to **0** = little or no linear relationship

We use this to:
1. Identify redundant features (highly correlated pairs carry similar information)
2. See which features are most directly related to the Target


In [ ]:
# ── Pearson correlation heatmap ───────────────────────────────────────────────
# We compute correlations only on numeric columns.
# cmap='coolwarm': blue = negative correlation, red = positive correlation.
# annot=True prints the actual correlation value inside each cell.

numeric_df = df_raw.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    numeric_df.corr(),
    annot=True,       # print numbers inside each cell
    fmt='.2f',        # round to 2 decimal places
    cmap='coolwarm',  # blue=negative, red=positive
    linewidths=0.5,   # thin separator lines between cells
    square=True,      # keep each cell square
    ax=ax,
)
ax.set_title('Pearson Correlation Matrix — Blood Donation Dataset', fontsize=14, pad=14)

fig.savefig(os.path.join(FIG_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/correlation_heatmap.png')


---
## Section 4 — Missing Value Analysis

Before cleaning, we quantify how much data is missing and where.  
Missing values must be handled before model training, because most ML algorithms
cannot process NaN values directly.


In [ ]:
# ── Count and percentage of missing values per column ─────────────────────────
# isnull().sum() gives the count of NaN values per column.
# Dividing by total rows and multiplying by 100 gives the percentage.

missing_count = df_raw.isnull().sum()
missing_pct   = (missing_count / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count'  : missing_count,
    'Missing Percent': missing_pct,
}).sort_values('Missing Percent', ascending=False)

print('=== Missing Value Report ===')
print(missing_report)
print(f'\nTotal missing cells : {missing_count.sum():,}')
print(f'Total cells         : {df_raw.size:,}')
print(f'Overall missing %   : {missing_count.sum() / df_raw.size * 100:.2f}%')


In [ ]:
# ── Visualize missing values as a horizontal bar chart ───────────────────────
# Only columns with at least one missing value are shown.
# Bar length = percentage of rows that are missing in that column.

raw_missing = df_raw.isnull().sum()
raw_missing = raw_missing[raw_missing > 0]

if raw_missing.empty:
    print('No missing values found in the raw dataset.')
    print('(A missing values chart will not be generated for this dataset.)')
else:
    fig, ax = plt.subplots(figsize=(8, max(3, len(raw_missing) * 0.5)))
    ax.barh(
        raw_missing.index,
        raw_missing.values / len(df_raw) * 100,
        color='#E74C3C',
        edgecolor='white',
    )
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Percentage by Column (Raw Data)', fontsize=12)

    fig.savefig(os.path.join(FIG_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to reports/figures/missing_values.png')


---
## Section 5 — Data Cleaning (KNN Imputation)

We now clean the raw dataset using the `DataCleaner` class from `src/data/cleaner.py`.

### Steps performed:
1. **Standardize missing markers** — replace strings like `'NA'`, `'--'`, `'?'` with `np.nan`
2. **Remove duplicate rows** — identical rows carry no new information
3. **KNN Imputation (k=5)** — estimate missing values from the 5 most similar records
4. **Encode categorical columns** — convert text labels (Blood Type, Location) to numbers

### Why KNN Imputation?
Mean/median imputation ignores relationships between features.  
KNN uses the `k` most similar rows (based on other features) to estimate a missing value,
preserving the local structure of the data. We use `k=5` as specified in the course guidelines.


In [ ]:
# ── Initialize the DataCleaner ────────────────────────────────────────────────
# n_neighbors=5 matches the course specification for KNN imputation.
from src.data.cleaner import DataCleaner

cleaner = DataCleaner(n_neighbors=5)

# Step 1: Replace placeholder strings ('NA', '--', '?', etc.) with np.nan
# This ensures pandas recognizes all missing values consistently.
df_standardized = cleaner.standardize_missing(df_raw.copy())

print('Missing values before standardization:', df_raw.isnull().sum().sum())
print('Missing values after  standardization:', df_standardized.isnull().sum().sum())


In [ ]:
# ── Step 2: Remove exact duplicate rows ───────────────────────────────────────
# Duplicate rows can bias the model by giving extra weight to repeated observations.
rows_before = len(df_standardized)
df_deduped  = cleaner.remove_duplicates(df_standardized)
rows_after  = len(df_deduped)

print(f'Rows before deduplication : {rows_before:,}')
print(f'Rows after  deduplication : {rows_after:,}')
print(f'Duplicate rows removed    : {rows_before - rows_after}')


In [ ]:
# ── Step 3: KNN Imputation (k=5) on numeric columns ──────────────────────────
# We first separate the target column (we NEVER impute the label itself).
# Categorical columns are label-encoded so KNN can compute numeric distances.
# KNNImputer then fills all remaining missing values.

y_series    = df_deduped[target_col].copy()
X_to_impute = df_deduped.drop(columns=[target_col]).copy()

# Label-encode any text/categorical columns before imputation
label_encoders = {}
for col in X_to_impute.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    X_to_impute[col] = X_to_impute[col].fillna('Unknown')
    X_to_impute[col] = le.fit_transform(X_to_impute[col].astype(str))
    label_encoders[col] = le

numeric_cols_impute = X_to_impute.select_dtypes(include=[np.number]).columns.tolist()
print(f'Columns to KNN-impute ({len(numeric_cols_impute)}): {numeric_cols_impute}')

missing_before = X_to_impute.isnull().sum().sum()
print(f'Missing values before KNN imputation: {missing_before:,}')

# Apply KNN imputation: estimates each missing value from 5 nearest neighbors
knn_imputer   = KNNImputer(n_neighbors=5)
X_imputed_arr = knn_imputer.fit_transform(X_to_impute)
X_imputed     = pd.DataFrame(X_imputed_arr, columns=X_to_impute.columns)

print(f'Missing values after  KNN imputation: {X_imputed.isnull().sum().sum()}')
print('KNN imputation complete — all missing values have been filled.')


---
## Section 6 — Feature Engineering

We create three new features from the existing RFMTC columns.  
These derived features may help the model better distinguish donors from non-donors.

| New Feature | Formula | Meaning |
|---|---|---|
| `donation_density` | Frequency / Time | Average donations per month |
| `recency_score` | 1 / (Recency + 1) | Higher = donated more recently |
| `high_frequency_flag` | 1 if Frequency > median | Binary flag for frequent donors |

After engineering, we save the clean dataset to `data/processed/`.


In [ ]:
# ── Import our FeatureBuilder class ───────────────────────────────────────────
from src.features.builder import FeatureBuilder

fb = FeatureBuilder(scale_method='standard')

# Add the three domain-specific derived features
X_engineered = fb.add_derived_features(X_imputed)

print(f'Features before engineering: {X_imputed.shape[1]}')
print(f'Features after  engineering: {X_engineered.shape[1]}')
new_features = [c for c in X_engineered.columns if c not in X_imputed.columns]
print(f'New feature names: {new_features}')


In [ ]:
# ── Combine features + target and save the clean dataset ──────────────────────
# We concatenate the engineered feature matrix with the original target series.
# reset_index(drop=True) ensures the row indices align correctly.

df_clean = pd.concat([X_engineered, y_series.reset_index(drop=True)], axis=1)

clean_path = os.path.join(PROC_DIR, 'blood_donation_clean.csv')
df_clean.to_csv(clean_path, index=False)

print(f'Clean dataset shape : {df_clean.shape}')
print(f'Saved to            : {clean_path}')
df_clean.head()


In [ ]:
# ── Confirm no missing values remain ──────────────────────────────────────────
remaining_nulls = df_clean.isnull().sum().sum()
print(f'Total missing values in clean dataset: {remaining_nulls}')

if remaining_nulls == 0:
    print('All missing values resolved. Dataset is ready for modeling.')
else:
    print('WARNING: Some missing values remain — review the cleaning steps.')


---
## Section 7 — Model Training (Random Forest Classifier)

We train a **Random Forest Classifier** on the cleaned, engineered dataset.

### Configuration (as specified in course guidelines):

| Parameter | Value | Reason |
|---|---|---|
| `n_estimators` | 100 | 100 decision trees in the forest |
| `random_state` | 42 | Reproducible results |
| `class_weight` | `'balanced'` | Auto-compensates for class imbalance |
| Train/test split | 80% / 20% | Standard academic practice |
| `stratify` | `y_all` | Ensures both splits have same class ratio |
| Scaling | `StandardScaler` | Zero-mean, unit-variance normalization |
| Cross-validation | 5-fold | Robust generalization estimate |


In [ ]:
# ── Prepare feature matrix (X) and target vector (y) ─────────────────────────
# StandardScaler normalizes features to have mean=0 and std=1.
# This ensures no single feature dominates due to its scale.

y_all = df_clean[target_col]
X_all = df_clean.drop(columns=[target_col])

scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_all), columns=X_all.columns)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'Target vector shape  : {y_all.shape}')
print(f'Features used: {X_all.columns.tolist()}')


In [ ]:
# ── Train/test split: 80% training, 20% testing (stratified) ─────────────────
# stratify=y_all ensures both splits preserve the same class proportions.
# random_state=42 makes the split reproducible.

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_all, test_size=0.20, random_state=42, stratify=y_all
)

print(f'Training set : {len(X_train):,} rows ({len(X_train)/len(X_scaled)*100:.0f}%)')
print(f'Test set     : {len(X_test):,} rows  ({len(X_test)/len(X_scaled)*100:.0f}%)')
print(f'\nClass balance in train: {y_train.value_counts().to_dict()}')
print(f'Class balance in test : {y_test.value_counts().to_dict()}')


In [ ]:
# ── Train the Random Forest Classifier ────────────────────────────────────────
# n_estimators=100 means 100 decision trees are built and combined.
# Each tree sees a random subset of rows and features — reducing overfitting.
# class_weight='balanced' adjusts loss weights inversely proportional to class frequency.

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,               # use all CPU cores
    class_weight='balanced'  # handle class imbalance
)

rf.fit(X_train, y_train)

print('Random Forest training complete.')
print(f'Train accuracy : {rf.score(X_train, y_train):.4f}')
print(f'Test  accuracy : {rf.score(X_test, y_test):.4f}')


In [ ]:
# ── 5-Fold Cross-Validation ────────────────────────────────────────────────────
# The dataset is split into 5 equal folds.
# The model trains on 4 folds and evaluates on the 5th — repeated 5 times.
# This gives a more reliable accuracy estimate than a single holdout split.

cv_scores = cross_val_score(rf, X_scaled, y_all, cv=5, scoring='accuracy', n_jobs=-1)

print('=== 5-Fold Cross-Validation Results ===')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\nMean accuracy : {cv_scores.mean():.4f}')
print(f'Std deviation : {cv_scores.std():.4f}')
print('\nA low std deviation means the model performs consistently across folds.')


---
## Section 8 — Model Evaluation

We evaluate the trained model on the **unseen test set** (20% of data held out).  
Using multiple metrics gives a complete picture of performance:

| Metric | What it measures |
|---|---|
| **Accuracy** | Overall % of correct predictions |
| **Precision** | Of predicted donors, how many actually donated |
| **Recall** | Of actual donors, how many we correctly identified |
| **F1 Score** | Harmonic mean of precision and recall |
| **ROC-AUC** | Overall ability to distinguish classes (1.0 = perfect) |


In [ ]:
# ── Generate predictions on the test set ──────────────────────────────────────
# y_pred = hard class labels (0 or 1) for each test record
# y_prob = probability of being class 1 (Will Donate) for each record

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# ── Compute all evaluation metrics ───────────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)
report_str = classification_report(
    y_test, y_pred,
    target_names=["Won't Donate", 'Will Donate']
)

print('=== Test Set Evaluation Metrics ===')
print(f'Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')
print('\n=== Full Classification Report ===')
print(report_str)


### 8.1 — Confusion Matrix

The confusion matrix shows the breakdown of correct and incorrect predictions.

|  | Predicted: Won't Donate | Predicted: Will Donate |
|---|---|---|
| **Actual: Won't Donate** | True Negative (TN) ✓ | False Positive (FP) ✗ |
| **Actual: Will Donate**  | False Negative (FN) ✗ | True Positive (TP) ✓ |

- **TN + TP** = correct predictions  
- **FP** = predicted 'will donate' but they won't (wasted outreach)
- **FN** = predicted 'won't donate' but they would (missed donor)


In [ ]:
# ── Confusion Matrix Visualization ────────────────────────────────────────────
# ConfusionMatrixDisplay creates an annotated heatmap of the confusion matrix.
# cmap='Blues': darker color = more predictions fell into that cell.

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    rf, X_test, y_test,
    display_labels=["Won't Donate", 'Will Donate'],
    cmap='Blues',
    ax=ax,
)
ax.set_title('Confusion Matrix — Random Forest', fontsize=12)

fig.savefig(os.path.join(FIG_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/confusion_matrix.png')


### 8.2 — ROC Curve (Receiver Operating Characteristic)

The ROC curve plots the **True Positive Rate** (Recall) against the **False Positive Rate**
as we vary the classification threshold from 0 to 1.

- A curve that hugs the **top-left corner** indicates strong performance
- **AUC** (Area Under the Curve) summarizes performance in one number (1.0 = perfect)
- The **dashed diagonal** line = a random classifier (AUC = 0.5)


In [ ]:
# ── ROC Curve ──────────────────────────────────────────────────────────────────
# roc_curve() computes the FPR and TPR at many different threshold values.
# We plot these as a curve and shade the area beneath it to visually show AUC.

fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#2980B9', lw=2, label=f'Random Forest (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#2980B9')  # shade the AUC region

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — Random Forest Classifier', fontsize=12)
ax.legend(loc='lower right')

fig.savefig(os.path.join(FIG_DIR, 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/roc_curve.png')


### 8.3 — Precision-Recall Curve

For imbalanced datasets, the PR curve is often more informative than the ROC curve.
It shows the trade-off between precision and recall at different thresholds.

- **High precision + high recall** = ideal model
- The **baseline** (dashed line) = proportion of positive class in the test set
- A curve **well above the baseline** means the model significantly outperforms random


In [ ]:
# ── Precision-Recall Curve ─────────────────────────────────────────────────────
# precision_recall_curve() computes precision and recall at many threshold values.
# The baseline represents a naive classifier that always predicts the majority class.

pr_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
baseline = y_test.mean()  # proportion of positive class in the test set

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec_vals, pr_vals, color='#2980B9', lw=2, label='Random Forest')
ax.axhline(y=baseline, color='gray', linestyle='--',
           label=f'Baseline (random): {baseline:.2f}')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontsize=12)
ax.legend()

fig.savefig(os.path.join(FIG_DIR, 'precision_recall_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/precision_recall_curve.png')


### 8.4 — Feature Importances

The Random Forest ranks each feature by how useful it was for making correct predictions.  
This is called **Gini Importance** — it measures how much each feature reduces impurity
across all 100 trees and all their splits.

A higher importance score means the model relied on that feature more heavily.


In [ ]:
# ── Feature Importance Chart ──────────────────────────────────────────────────
# rf.feature_importances_ returns one importance score per feature.
# We sort by importance and plot as a horizontal bar chart.
# The blue gradient (darker = more important) makes the ranking visually clear.

fi_df = pd.DataFrame({
    'feature'   : X_all.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('=== Feature Importances (sorted, highest first) ===')
print(fi_df.to_string(index=False))

# Sort ascending so the most important feature appears at the TOP of the chart
top_fi = fi_df.sort_values('importance')
colors = plt.cm.Blues(np.linspace(0.35, 0.9, len(top_fi)))

fig, ax = plt.subplots(figsize=(8, max(4, len(top_fi) * 0.4)))
ax.barh(top_fi['feature'], top_fi['importance'], color=colors, edgecolor='white')
ax.set_xlabel('Gini Importance')
ax.set_title(f'Feature Importances — Random Forest ({len(top_fi)} features)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

fig.savefig(os.path.join(FIG_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/figures/feature_importance.png')


---
## Section 9 — Summary of Results

This section brings together all key findings from the analysis in one place,  
suitable for presenting to your instructor.


In [ ]:
# ── Final results summary ─────────────────────────────────────────────────────
print('=' * 60)
print('  BLOOD DONATION PREDICTION — FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'\n  Dataset        : {len(df_clean):,} records, {X_all.shape[1]} features')
print(f'  Target column  : {target_col}')
print(f'  Positive class : Will Donate (1)  — {y_all.sum():,} records ({y_all.mean()*100:.1f}%)')
print(f"  Negative class : Won't Donate (0) — {(y_all==0).sum():,} records ({(y_all==0).mean()*100:.1f}%)")
print(f'\n  Algorithm      : Random Forest Classifier')
print(f'  n_estimators   : 100')
print(f'  class_weight   : balanced')
print(f'  Imputation     : KNN (k=5)')
print(f'  Scaling        : StandardScaler')
print(f'\n  --- Test Set Metrics (20% holdout) ---')
print(f'  Accuracy       : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Precision      : {prec:.4f}')
print(f'  Recall         : {rec:.4f}')
print(f'  F1 Score       : {f1:.4f}')
print(f'  ROC-AUC        : {auc:.4f}')
print(f'\n  --- Cross-Validation (5-fold) ---')
print(f'  CV Mean Acc    : {cv_scores.mean():.4f}')
print(f'  CV Std         : {cv_scores.std():.4f}')
print(f'\n  Top-3 Features :')
for idx, row in fi_df.head(3).iterrows():
    print(f'    {int(idx)+1}. {row["feature"]:<25} Importance: {row["importance"]:.4f}')
print(f'\n  --- Output Files ---')
print(f'  Figures saved to  : reports/figures/ (9 PNG files)')
print(f'  Clean data saved  : data/processed/blood_donation_clean.csv')
print('=' * 60)


In [ ]:
# ── Key Takeaways ─────────────────────────────────────────────────────────────
# These bullet points summarize the most important findings from the analysis.
print('Key Findings:')
print(f'  1. The Random Forest achieves {acc*100:.1f}% accuracy on unseen test data.')
print(f'  2. ROC-AUC of {auc:.3f} indicates strong ability to distinguish donors from non-donors.')
print(f'  3. {fi_df.iloc[0]["feature"]} is the single most important predictor of donation behavior.')
print( '  4. KNN imputation (k=5) resolved all missing values without distributional bias.')
print(f'  5. 5-fold CV shows stable performance (std = {cv_scores.std():.4f}).')
print( '  6. All 9 charts are displayed inline above and saved to reports/figures/.')
